In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import torch

#flatten before conversion:
# transform to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)


In [ ]:
# 2. Create TensorDataset objects
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)



In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset ,batch_size=32 , shuffle=True)  # <Replace None with your code>
test_loader = DataLoader(test_dataset, batch_size=32 , shuffle=False)   # <Replace None with your code>



In [ ]:
# 4. Print shape of one batch

first_sample, _ = train_dataset[0]
print(f"Shape of one sample: {first_sample.shape}")

In [ ]:
# 5. Display sample images

#idk tbh

In [ ]:
# Task 1: Write your model class here:


class NN4Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim):
    super(NN4Layer, self).__init__()
    # TODO: Define the first linear layer: input_dim -> hidden_dim
    self.layer1 = nn.Linear(in_features=input_dim , out_features=hidden_dim)  # <Replace None with your code>

    # TODO: Define the second linear layer: hidden_dim -> hidden_dim
    self.layer2 = nn.Linear(in_features=hidden_dim, out_features=hidden_dim)  # <Replace None with your code>

    # TODO: Define the 3rd layer: hidden_dim -> 1 (single value for regression)
    self.layer3 = nn.Linear(in_features=hidden_dim, out_features= 1)  # <Replace None with your code>

    #4th layer
    self.layer4 = nn.Linear(in_features=input_dim,out_features=hidden_dim)

    # TODO: Define ReLU activation
    self.relu = nn.ReLU()  # <Replace None with your code>

  def forward(self, x):
    # TODO: First hidden layer with ReLU
    a1 = self.relu(self.layer1(x))  # <Replace None with your code>

    # TODO: Second hidden layer with ReLU
    a2 = self.relu(self.layer2(a1))  # <Replace None with your code>

    # TODO: 3rd layer
    a3 = self.relu(a2)  # <Replace None with your code>

    #4th layer:
    output = self.relu(a3)


    return output

In [ ]:
# Task 2: Write your training loop here:


def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # TODO: Set the model to training mode
  model.train()

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # TODO: Move batch to the selected device
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device).reshape(-1, 1)  # Reshape for consistency with output

    # TODO: Forward pass - get model predictions
    outputs = model(X_batch)

    # TODO: Compute loss using criterion
    loss = criterion(outputs, y_batch)  # Use criterion directly

    # TODO: Backward pass & optimization
    # Step 1: Clear previous gradients
    # If we don't do this, gradients accumulate from previous batches!
    optimizer.zero_grad()

    # Step 2: Compute gradients (backward pass)
    # Calculates how much each weight contributed to the error
    loss.backward()

    # Step 3: Update model parameters
    # Adjusts the weights based on the gradients
    optimizer.step()
    running_loss = running_loss + loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, criterion, test_loader, device):
  # TODO: Set the model to evaluation mode
  # <YOUR CODE HERE>
  model.eval()

  running_loss = 0.0

  # TODO: Disable gradient computation using torch.no_grad()
  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      # TODO: Move data to device
      X_batch = X_batch.to(device)  # <Replace None with your code>

      # <Replace None with your code> [HINT: reshape to (-1, 1)]
      y_batch = y_batch.to(device).reshape(-1, 1)

      # TODO: Forward pass - get model predictions
      # <Replace None with your code>
      outputs = model(X_batch)

      # TODO: Compute loss using criterion
      # <Replace None with your code>
      loss = criterion(outputs, y_batch)

      # Accumulate loss

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:

# TODO: Set up the device (use GPU if available, otherwise CPU)
# <Replace None with your code>
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Model parameters
input_dim = X_train.shape[1]   # Number of tabular features
hidden_dim = 64                # Design choice (feel free to experiment!)

# TODO: Instantiate the model and move it to the device
# <Replace None with your code>
model = NN4Layer(input_dim, hidden_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")



# TODO: Define criterion (loss function) - use MSELoss for regression
# <Replace None with your code>
criterion = nn.MSELoss()


# Hyperparameters (feel free to experiment!)
num_epochs = 20
learning_rate = 0.001


# TODO: Define optimizer - use AdamW with the model parameters and learning rate


from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=learning_rate)

In [ ]:
# Task 5: Start training for 20 epochs:

# Run Training
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  # TODO: Train one epoch using the train_one_epoch function
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # Validate using the validate function
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:



# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test.numpy(), predictions.flatten(), alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), predictions.min())
max_val = max(y_test.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($)', fontsize=12)
plt.ylabel('Predicted Price ($)', fontsize=12)
plt.title('Predicted vs Actual Diamond Prices', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()